# Stage 03 — Primary Model Development

## Stage Overview
This stage verifies the frozen source package, reconstructs the locked decontaminated BanglaBait view, enforces D0, freezes cue extraction, performs TRAIN-only configuration development, and refits the four locked models on full CLEAN TRAIN.

## Scientific Objective
H3 remains the primary scientific target downstream. Stage 03 does not evaluate H1–H4 and does not access external labels for model decisions. DEV and TEST may be read only for D0 integrity/decontamination auditing.


## Input Contract

### Required Input
One Kaggle Dataset containing the frozen Swarabyanjan external-data package.

The notebook automatically detects either:

1. a literal `EXTERNAL-DATASET-BYTES-v1.zip`, or
2. Kaggle's extracted frozen directory tree, including nested wrappers such as `EXTERNAL-DATASET-BYTES-v1/EXTERNAL-DATASET-BYTES-v1/`.

Manual path configuration, manual unzipping/rezipping, and dataset-slug editing are not required.

### Runtime
Kaggle Notebook with Internet enabled for immutable Hugging Face revision resolution and first-time model download.

### Accelerator
**GPU required** for the complete Stage 03 experiment because BanglaBERT and XLM-R training are mandatory.

### Pipeline Position
03 / 04 / 05 / 06

### Upstream Requirements
No generated upstream artifact is required. Stage 01 and Stage 02 identities are embedded as frozen provenance constants.

### Read-only Inputs
The discovered frozen package root and all files beneath it are immutable. In ZIP mode, extraction occurs only in temporary working storage; in extracted-directory mode, the Kaggle-mounted directory is read directly.

### Generated Outputs
`/kaggle/working/03_results/03_primary_model_development/`, including frozen-input discovery metadata, clean-view manifests, D0 reports, cue schema, TRAIN-only split manifest, model/configuration locks, model states, hashes, and runtime metadata.

### Manual Uploads
No Stage 01 notebook, manually cleaned BanglaBait splits, literature files, intermediate datasets, or separately downloaded external files are required.


## Environment Validation
The Kaggle compiled scientific stack is preserved. The notebook does not upgrade NumPy, SciPy, pandas, scikit-learn, pyarrow, or PyTorch. Missing application-level packages are installed under exact constraints that pin the existing compiled stack, followed by a fresh-process import/API probe.


In [1]:
# Environment preflight — standard library only.
# The compiled Kaggle scientific stack is preserved. Only missing application-level packages may be installed.
from pathlib import Path
import importlib.metadata as _md
import importlib.util as _util
import json as _json
import os as _os
import subprocess as _subprocess
import sys as _sys
import tempfile as _tempfile
import textwrap as _textwrap

STAGE_ENVIRONMENT_ID = "SWARABYANJAN-KAGGLE-ENV-v2.0.0"
CORE_DISTRIBUTIONS = {
    "numpy":"numpy",
    "scipy":"scipy",
    "pandas":"pandas",
    "scikit-learn":"sklearn",
    "pyarrow":"pyarrow",
    "torch":"torch",
    "statsmodels":"statsmodels",
    "joblib":"joblib",
}
APPLICATION_REQUIREMENTS = [('interpret-core', 'interpret', 'interpret-core>=0.6,<0.8'), ('transformers', 'transformers', 'transformers>=4.45,<6'), ('accelerate', 'accelerate', 'accelerate>=0.30,<2'), ('sentencepiece', 'sentencepiece', 'sentencepiece>=0.1.99,<0.3')]

def _dist_version(name):
    try:
        return _md.version(name)
    except _md.PackageNotFoundError:
        return None

def _module_present(name):
    try:
        return _util.find_spec(name) is not None
    except Exception:
        return False

initial_versions = {name:_dist_version(name) for name in list(CORE_DISTRIBUTIONS)+[x[0] for x in APPLICATION_REQUIREMENTS]}
missing_core = [dist for dist,module in CORE_DISTRIBUTIONS.items() if not _module_present(module)]
if missing_core:
    raise RuntimeError(
        "WHAT FAILED: the standard Kaggle compiled scientific stack is incomplete. "
        f"Missing core distributions/modules: {missing_core}. "
        "WHY: installing or replacing NumPy/SciPy/pandas/scikit-learn/pyarrow/torch inside this notebook can create ABI conflicts. "
        "SAFE TO CONTINUE: NO. Start a fresh standard Kaggle Python runtime; do not upgrade the scientific stack."
    )

missing_apps = [(dist,module,spec) for dist,module,spec in APPLICATION_REQUIREMENTS if not _module_present(module)]
installed_now = []
if missing_apps:
    # Pin every existing compiled/core package so the dependency resolver cannot replace Kaggle's ABI-compatible stack.
    constraints = []
    for dist in CORE_DISTRIBUTIONS:
        v = _dist_version(dist)
        if v:
            constraints.append(f"{dist}=={v}")
    cfile = Path(_tempfile.gettempdir()) / f"swarabyanjan_03_constraints.txt"
    cfile.write_text("\n".join(constraints)+"\n", encoding="utf-8")
    specs = [spec for _,_,spec in missing_apps]
    cmd = [
        _sys.executable, "-m", "pip", "install",
        "--disable-pip-version-check", "--no-input", "--prefer-binary",
        "--constraint", str(cfile), *specs
    ]
    proc = _subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(
            "WHAT FAILED: installation of missing application-level dependencies failed. "
            f"Requested: {specs}. "
            "WHY: pip could not satisfy them without changing the pinned Kaggle scientific stack or network/package access failed. "
            "SAFE TO CONTINUE: NO.\n"
            f"pip stdout:\n{proc.stdout[-4000:]}\n"
            f"pip stderr:\n{proc.stderr[-4000:]}"
        )
    installed_now = specs

# Import/API validation occurs in a fresh subprocess before compiled packages are imported into this notebook kernel.
_probe = r"""
import json
import numpy, scipy, pandas, sklearn, pyarrow, torch, statsmodels, joblib
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from interpret.glassbox import ExplainableBoostingClassifier
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from huggingface_hub import model_info
print(json.dumps({
 "numpy":numpy.__version__,"scipy":scipy.__version__,"pandas":pandas.__version__,
 "sklearn":sklearn.__version__,"pyarrow":pyarrow.__version__,"torch":torch.__version__,
 "statsmodels":statsmodels.__version__
}, sort_keys=True))
"""
probe_proc = _subprocess.run([_sys.executable, "-c", _probe], capture_output=True, text=True)
if probe_proc.returncode != 0:
    raise RuntimeError(
        "WHAT FAILED: post-setup import/API validation failed. "
        "WHY: the runtime is still incompatible or a required API is unavailable. "
        "SAFE TO CONTINUE: NO. Restart with a fresh Kaggle Python runtime rather than upgrading the scientific stack.\n"
        f"probe stdout:\n{probe_proc.stdout[-4000:]}\n"
        f"probe stderr:\n{probe_proc.stderr[-4000:]}"
    )

ENVIRONMENT_PREFLIGHT = {
    "schema_version": 1,
    "environment_id": STAGE_ENVIRONMENT_ID,
    "stage_id": "03",
    "policy": "preserve Kaggle compiled scientific stack; install only missing application-level packages",
    "accelerator_note": 'GPU required for complete Stage 03',
    "initial_versions": initial_versions,
    "installed_now": installed_now,
    "validated_versions": {name:_dist_version(name) for name in list(CORE_DISTRIBUTIONS)+[x[0] for x in APPLICATION_REQUIREMENTS]},
    "probe_stdout": probe_proc.stdout.strip(),
}
print("Environment validation: PASS")
print(probe_proc.stdout.strip())


Environment validation: PASS
{"numpy": "2.0.2", "pandas": "2.3.3", "pyarrow": "24.0.0", "scipy": "1.16.3", "sklearn": "1.6.1", "statsmodels": "0.14.6", "torch": "2.10.0+cu128"}


## Reproducibility and Runtime Configuration
Runtime identity, package versions, git state when available, deterministic seeds, and output hashes are recorded before downstream use.


In [2]:
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
from collections import Counter, defaultdict
import csv, hashlib, importlib.metadata as im, inspect, json, math, os, platform, random, re
import shutil, subprocess, sys, unicodedata, zipfile

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from scipy.special import expit
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
import joblib

STAGE_ID = "03"
STAGE_NAME = "03_primary_model_development"
PROTOCOL_VERSION = "SWARABYANJAN-CROSSPLATFORM-v1.0.0"
DECONTAM_VERSION = "BANGLABAIT-PROTECTED-TITLE-DECONTAM-v1.0.0"
CUE_VERSION = "BANGLA-CLICKBAIT-CUES-v1.0.0"
P0_METHOD = "P0-v1.1.1"
FREEZE_ID = "EXTERNAL-DATASET-BYTES-v1"
FROZEN_ZIP_NAME = "EXTERNAL-DATASET-BYTES-v1.zip"
EXPECTED_ZIP_SHA256 = "e0de5c224f406fe22d8ad30804217465f789c49c70e1d3afbe00afd28deb7ccf"
EXPECTED_STAGE01_NOTEBOOK_SHA256 = "627e38c7fd99c53b483bc109404b03e781b060c58215e639121838cb63abe8cf"
AUTHORITATIVE_STAGE02_METHODOLOGY_SHA256 = "fcf658785c8a91ffa31a08b7861dbed07aac26f06b52d133ce257f3610d3ce6b"
AUTHORITATIVE_STAGE02_NOTEBOOK_SHA256 = "be59c43013c20e052bc9475954303a75fed8807dfed689ef517590b382b0214b"
AUTHORITATIVE_STAGE02_DECISION = "P0_NO_GO"
EXPECTED_BANGLABAIT_LABEL_DOMAIN = ("0", "1")
SEEDS = [42, 137, 2718, 31415, 65537]
CONFIG_SPLIT_SEED = 20260906
HF_MODELS = {
    "banglabert": "csebuetnlp/banglabert",
    "xlmr": "FacebookAI/xlm-roberta-base",
}
MODEL_KEYS = ["ebm", "tfidf_lr", "banglabert", "xlmr"]

INPUT_ROOT = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path.cwd()
WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUT_ROOT = WORK_ROOT / "03_results" / STAGE_NAME
TEMP_ROOT = WORK_ROOT / f".{STAGE_NAME}_tmp"

if OUT_ROOT.exists() and any(OUT_ROOT.iterdir()):
    raise RuntimeError(f"Refusing to overwrite existing Stage-03 output directory: {OUT_ROOT}")
OUT_ROOT.mkdir(parents=True, exist_ok=True)
TEMP_ROOT.mkdir(parents=True, exist_ok=True)


def utc_now():
    return datetime.now(timezone.utc).isoformat()


def canonical_json_bytes(obj):
    return json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode("utf-8")


def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()


def sha256_file(path, chunk_size=8*1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                return h.hexdigest()
            h.update(b)


def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, sort_keys=True, indent=2) + "\n", encoding="utf-8", newline="\n")
    tmp.replace(path)


def write_text(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(text, encoding="utf-8", newline="\n")
    tmp.replace(path)


def set_status(status, message, details=None):
    payload = {"stage_id": STAGE_ID, "stage_name": STAGE_NAME, "status": status, "message": message,
               "timestamp_utc": utc_now(), "details": details or {}}
    write_json(OUT_ROOT / "STAGE_STATUS.json", payload)
    print(f"STAGE {STAGE_ID} STATUS: {status} — {message}")


def hard_stop(message, details=None):
    set_status("HARD_STOP", message, details)
    raise RuntimeError(message)


def package_version(name):
    try:
        return im.version(name)
    except Exception:
        return None


def git_state():
    state = {"available": False, "head": None, "dirty": None, "status_porcelain": None}
    try:
        p = subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True, timeout=10)
        if p.returncode != 0:
            return state
        root = p.stdout.strip()
        head = subprocess.run(["git", "-C", root, "rev-parse", "HEAD"], capture_output=True, text=True, timeout=10, check=True).stdout.strip()
        st = subprocess.run(["git", "-C", root, "status", "--porcelain"], capture_output=True, text=True, timeout=10, check=True).stdout
        state.update({"available": True, "root": root, "head": head, "dirty": bool(st.strip()), "status_porcelain": st.splitlines()})
    except Exception as e:
        state["error"] = f"{type(e).__name__}: {e}"
    return state


def environment_record():
    return {
        "timestamp_utc": utc_now(), "python": sys.version, "platform": platform.platform(),
        "cpu_count": os.cpu_count(), "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
        "packages": {name: package_version(name) for name in [
            "pandas","numpy","pyarrow","openpyxl","scikit-learn","interpret-core","transformers",
            "torch","accelerate","huggingface-hub","scipy","statsmodels","joblib"]},
        "git": git_state(),
    }

write_json(OUT_ROOT / "ENVIRONMENT_PREFLIGHT.json", ENVIRONMENT_PREFLIGHT)
set_status("RUNNING", "Stage-03 initialization complete")


STAGE 03 STATUS: RUNNING — Stage-03 initialization complete


## Dataset Loading, Integrity, and D0 Gate
The canonical ZIP is discovered recursively under the Kaggle input mount and verified against the frozen identity. Decontamination uses the locked P0 normalization and complete-equivalence-class quarantine rule. Any D0 failure stops before model fitting.


In [3]:

# ---------- Frozen Stage-01 package discovery and verification ----------
# Supports both a literal pinned ZIP and Kaggle's automatically extracted directory tree.
FROZEN_REQUIRED_CONTROLS = (
    "ACQUISITION_MANIFEST.json",
    "ACQUISITION_STATUS.json",
    "SOURCES_LOCK.json",
    "SHA256SUMS.txt",
    "README_HANDOFF.txt",
)
FROZEN_REQUIRED_DATASET_DIRS = (
    "BaitBuster-Bangla",
    "BanFakeNews-2.0",
    "BanMANI",
    "BanglaBait",
    "BanglaClick",
)

class FrozenPackageDiscoveryError(RuntimeError):
    pass


def safe_extract_zip(zip_path, destination):
    zip_path = Path(zip_path)
    destination = Path(destination).resolve()
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        for info in zf.infolist():
            target = (destination / info.filename).resolve()
            if target != destination and destination not in target.parents:
                raise FrozenPackageDiscoveryError(f"Unsafe ZIP member: {info.filename}")
        bad = zf.testzip()
        if bad is not None:
            raise FrozenPackageDiscoveryError(f"ZIP CRC verification failed at member: {bad}")
        zf.extractall(destination)


def parse_sha256sums(path):
    out = {}
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        parts = line.split(None, 1)
        if len(parts) != 2 or not re.fullmatch(r"[0-9a-fA-F]{64}", parts[0]):
            raise FrozenPackageDiscoveryError(f"Malformed SHA256SUMS.txt line: {line!r}")
        rel = parts[1].lstrip("*").strip().replace("\\", "/")
        pp = PurePosixPath(rel)
        if pp.is_absolute() or ".." in pp.parts:
            raise FrozenPackageDiscoveryError(f"Unsafe path in SHA256SUMS.txt: {rel}")
        if rel in out:
            raise FrozenPackageDiscoveryError(f"Duplicate path in SHA256SUMS.txt: {rel}")
        out[rel] = parts[0].lower()
    if not out:
        raise FrozenPackageDiscoveryError("SHA256SUMS.txt is empty")
    return out


def _checksum_target_variants(root, rel):
    """Return safe candidate paths, tolerating one or more repeated package-name prefixes."""
    root = Path(root).resolve()
    pp = PurePosixPath(rel)
    parts = list(pp.parts)
    variants = [parts[:]]
    stripped = parts[:]
    while stripped and stripped[0] in {root.name, FREEZE_ID}:
        stripped = stripped[1:]
        if stripped:
            variants.append(stripped[:])
    out = []
    seen = set()
    for vp in variants:
        p = root.joinpath(*vp).resolve()
        if p != root and root not in p.parents:
            continue
        if p not in seen:
            seen.add(p)
            out.append(p)
    return out


def validate_frozen_root(root):
    """Validate one extracted package root and return a content-derived package identity."""
    root = Path(root).resolve()
    missing_controls = [name for name in FROZEN_REQUIRED_CONTROLS if not (root / name).is_file()]
    missing_dirs = [name for name in FROZEN_REQUIRED_DATASET_DIRS if not (root / name).is_dir()]
    if missing_controls or missing_dirs:
        raise FrozenPackageDiscoveryError(
            f"Structural markers missing at {root}: controls={missing_controls}, dataset_dirs={missing_dirs}"
        )

    checksums = parse_sha256sums(root / "SHA256SUMS.txt")
    failures = []
    for rel, expected in sorted(checksums.items()):
        candidates = [p for p in _checksum_target_variants(root, rel) if p.is_file()]
        if not candidates:
            failures.append({"path": rel, "reason": "missing"})
            continue
        # Repeated wrapper-prefix variants may resolve to more than one existing path only in malformed trees.
        actuals = [(p, sha256_file(p)) for p in candidates]
        matches = [(p, h) for p, h in actuals if h == expected]
        if not matches:
            failures.append({
                "path": rel, "reason": "sha256_mismatch",
                "expected": expected, "actuals": [(str(p), h) for p, h in actuals]
            })
    if failures:
        raise FrozenPackageDiscoveryError(
            f"File-level checksum verification failed at {root}: {failures[:10]}"
        )

    control_hashes = {name: sha256_file(root / name) for name in FROZEN_REQUIRED_CONTROLS}
    identity_payload = {
        "schema_version": 1,
        "freeze_id": FREEZE_ID,
        "controls": control_hashes,
        "declared_sha256sums": sorted(checksums.items()),
        "dataset_dirs": sorted(FROZEN_REQUIRED_DATASET_DIRS),
    }
    package_identity = hashlib.sha256(
        json.dumps(identity_payload, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode("utf-8")
    ).hexdigest()
    return {
        "root": root,
        "package_identity": package_identity,
        "checksum_entry_count": len(checksums),
        "control_hashes": control_hashes,
        "checksums": checksums,
    }


def _find_structural_roots(search_roots):
    candidates = set()
    for search_root in search_roots:
        search_root = Path(search_root)
        if not search_root.exists():
            continue
        try:
            markers = search_root.rglob("SOURCES_LOCK.json")
        except Exception:
            continue
        for marker in markers:
            root = marker.parent.resolve()
            if all((root / name).is_file() for name in FROZEN_REQUIRED_CONTROLS[:4]):
                candidates.add(root)
    return sorted(candidates, key=str)


def discover_frozen_package(search_roots, temp_root, expected_zip_sha256=EXPECTED_ZIP_SHA256,
                            zip_name=FROZEN_ZIP_NAME):
    """
    Discover a canonical frozen package in either mode:
      ZIP: literal EXTERNAL-DATASET-BYTES-v1.zip with the pinned ZIP SHA.
      EXTRACTED_DIRECTORY: structurally valid extracted package root.

    Identical content identities are deduplicated. Multiple distinct valid identities hard-stop.
    """
    search_roots = [Path(p) for p in search_roots if Path(p).exists()]
    temp_root = Path(temp_root)
    temp_root.mkdir(parents=True, exist_ok=True)

    valid = []
    invalid = []

    # Mode B first: directly mounted/extracted Kaggle package tree.
    for root in _find_structural_roots(search_roots):
        try:
            meta = validate_frozen_root(root)
            valid.append({
                "mode": "EXTRACTED_DIRECTORY",
                "source_path": root,
                **meta,
                "zip_sha256": None,
            })
        except Exception as exc:
            invalid.append({"mode": "EXTRACTED_DIRECTORY", "path": str(root), "error": f"{type(exc).__name__}: {exc}"})

    # Mode A: literal canonical ZIP, if mounted.
    zip_paths = set()
    for search_root in search_roots:
        try:
            for p in search_root.rglob(zip_name):
                if p.is_file():
                    zip_paths.add(p.resolve())
        except Exception:
            pass

    for idx, zp in enumerate(sorted(zip_paths, key=str)):
        try:
            zhash = sha256_file(zp)
            if zhash != expected_zip_sha256:
                raise FrozenPackageDiscoveryError(
                    f"ZIP SHA-256 mismatch for {zp}: expected {expected_zip_sha256}, observed {zhash}"
                )
            extract_dir = temp_root / f"zip_{idx:02d}_{zhash[:12]}"
            safe_extract_zip(zp, extract_dir)
            roots = _find_structural_roots([extract_dir])
            if not roots:
                raise FrozenPackageDiscoveryError("ZIP contains no structurally valid frozen package root")
            local_valid = []
            for eroot in roots:
                try:
                    local_valid.append(validate_frozen_root(eroot))
                except Exception:
                    pass
            if len(local_valid) != 1:
                raise FrozenPackageDiscoveryError(
                    f"ZIP must resolve to exactly one valid package root; found {len(local_valid)}"
                )
            meta = local_valid[0]
            valid.append({
                "mode": "ZIP",
                "source_path": zp,
                **meta,
                "zip_sha256": zhash,
            })
        except Exception as exc:
            invalid.append({"mode": "ZIP", "path": str(zp), "error": f"{type(exc).__name__}: {exc}"})

    if not valid:
        raise FrozenPackageDiscoveryError(
            "No valid frozen Swarabyanjan package was found. "
            "Accepted inputs are either the literal pinned ZIP or an extracted directory containing "
            "the Stage-01 control manifests and expected dataset directories. "
            f"Rejected candidates: {invalid[:20]}"
        )

    by_identity = defaultdict(list)
    for item in valid:
        by_identity[item["package_identity"]].append(item)

    if len(by_identity) > 1:
        conflict = {
            identity: [
                {"mode": x["mode"], "source_path": str(x["source_path"]), "root": str(x["root"])}
                for x in items
            ]
            for identity, items in sorted(by_identity.items())
        }
        raise FrozenPackageDiscoveryError(
            "Multiple genuinely conflicting frozen package identities were discovered: "
            + json.dumps(conflict, ensure_ascii=False, sort_keys=True)
        )

    identity, identical = next(iter(by_identity.items()))
    # Prefer a read-only extracted Kaggle tree when both representations contain identical bytes.
    identical = sorted(
        identical,
        key=lambda x: (0 if x["mode"] == "EXTRACTED_DIRECTORY" else 1, str(x["source_path"]))
    )
    selected = identical[0]
    selected = dict(selected)
    selected["duplicate_equivalent_candidates"] = [
        {"mode": x["mode"], "source_path": str(x["source_path"]), "root": str(x["root"])}
        for x in identical[1:]
    ]
    selected["invalid_candidates"] = invalid
    return selected


search_roots = [INPUT_ROOT]
if Path.cwd().resolve() != INPUT_ROOT.resolve():
    search_roots.append(Path.cwd())

try:
    FROZEN_DISCOVERY = discover_frozen_package(
        search_roots=search_roots,
        temp_root=TEMP_ROOT / "frozen_input_discovery",
    )
except FrozenPackageDiscoveryError as exc:
    hard_stop(
        "Frozen external-data package discovery failed",
        {
            "what_failed": str(exc),
            "why": "Neither a valid pinned ZIP nor a structurally/checksum-valid extracted frozen package could be selected unambiguously.",
            "safe_to_continue": False,
        },
    )

FROZEN_ROOT = Path(FROZEN_DISCOVERY["root"]).resolve()
FROZEN_INPUT_MODE = FROZEN_DISCOVERY["mode"]
FROZEN_INPUT_SOURCE = Path(FROZEN_DISCOVERY["source_path"]).resolve()
FROZEN_PACKAGE_IDENTITY = FROZEN_DISCOVERY["package_identity"]
checksums = FROZEN_DISCOVERY["checksums"]

SOURCES_LOCK = json.loads((FROZEN_ROOT / "SOURCES_LOCK.json").read_text(encoding="utf-8"))
lock_text = json.dumps(SOURCES_LOCK, ensure_ascii=False, sort_keys=True)
for token in [
    "ff22fb3b423952959a80dc5d9c6c3345b21cd145",
    "2ba28763d8f80eaba7a649877eae1fbbcb78a0e7",
    "be1528ba40dd8828c5c06ee70b7574cca22b8ee1",
]:
    if token not in lock_text:
        hard_stop("Frozen source-identity token absent from SOURCES_LOCK.json", {"token": token})

SOURCE_PATHS = {
    "train": FROZEN_ROOT / "BanglaBait/repository_snapshot/train.csv",
    "dev": FROZEN_ROOT / "BanglaBait/repository_snapshot/dev.csv",
    "test": FROZEN_ROOT / "BanglaBait/repository_snapshot/test.csv",
}
for split, p in SOURCE_PATHS.items():
    if not p.is_file():
        hard_stop("Protected BanglaBait split is missing", {"split": split, "path": str(p)})

INPUT_IDENTITY = {
    "freeze_id": FREEZE_ID,
    "input_mode": FROZEN_INPUT_MODE,
    "input_source_path": str(FROZEN_INPUT_SOURCE),
    "frozen_root": str(FROZEN_ROOT),
    "content_identity_sha256": FROZEN_PACKAGE_IDENTITY,
    "canonical_zip_sha256_provenance": EXPECTED_ZIP_SHA256,
    "observed_zip_sha256": FROZEN_DISCOVERY.get("zip_sha256"),
    "duplicate_equivalent_candidates": FROZEN_DISCOVERY.get("duplicate_equivalent_candidates", []),
    "stage01_notebook_sha256_historical": EXPECTED_STAGE01_NOTEBOOK_SHA256,
    "authoritative_stage02_methodology_sha256": AUTHORITATIVE_STAGE02_METHODOLOGY_SHA256,
    "authoritative_stage02_notebook_sha256": AUTHORITATIVE_STAGE02_NOTEBOOK_SHA256,
    "authoritative_stage02_decision": AUTHORITATIVE_STAGE02_DECISION,
    "sha256sum_entry_count": len(checksums),
}
write_json(OUT_ROOT / "INPUT_IDENTITY.json", INPUT_IDENTITY)
write_json(OUT_ROOT / "FROZEN_INPUT_DISCOVERY.json", {
    "schema_version": 1,
    "mode": FROZEN_INPUT_MODE,
    "source_path": str(FROZEN_INPUT_SOURCE),
    "root": str(FROZEN_ROOT),
    "content_identity_sha256": FROZEN_PACKAGE_IDENTITY,
    "canonical_zip_sha256_provenance": EXPECTED_ZIP_SHA256,
    "observed_zip_sha256": FROZEN_DISCOVERY.get("zip_sha256"),
    "duplicate_equivalent_candidates": FROZEN_DISCOVERY.get("duplicate_equivalent_candidates", []),
    "invalid_candidates": FROZEN_DISCOVERY.get("invalid_candidates", []),
})
print("Frozen input verification: PASS —", f"mode={FROZEN_INPUT_MODE}", f"root={FROZEN_ROOT}")


Frozen input verification: PASS — mode=EXTRACTED_DIRECTORY root=/kaggle/input/datasets/smalakarishere/swarabyanjan-frozen-dataset-v1/EXTERNAL-DATASET-BYTES-v1/EXTERNAL-DATASET-BYTES-v1


In [4]:
# ---------- Exact P0-v1.1.1 protected-title semantics ----------
PROTECTED_TITLE_PLACEHOLDERS = {
    "-", "--", "---", "_", "__", "missing", "n/a", "na", "nan", "no title", "none", "null", "unknown", "untitled"
}


def normalize_text_p0(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ""
    text = str(value)
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u200b", "").replace("\u200c", "").replace("\u200d", "").replace("\ufeff", "")
    text = text.casefold().strip()
    text = re.sub(r"\s+", " ", text)
    return text


def canonicalize_label_p0(raw_label):
    if raw_label is None:
        return ""
    return unicodedata.normalize("NFKC", str(raw_label)).strip()


def sha_text(text):
    return hashlib.sha256(str(text).encode("utf-8")).hexdigest()


def classify_title(raw_title):
    if raw_title is None:
        return "MISSING"
    norm = normalize_text_p0(raw_title)
    if norm == "":
        return "EMPTY"
    if norm in PROTECTED_TITLE_PLACEHOLDERS:
        return "KNOWN_PLACEHOLDER"
    if not any(ch.isalnum() for ch in norm):
        return "NON_TEXTUAL_GARBAGE"
    return "VALID_SHORT" if len(norm) < 8 else "VALID"


def load_protected_split(path, split, artifact_id):
    records = []
    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        headers = [str(x) for x in (reader.fieldnames or [])]
        if "title" not in headers or "label" not in headers:
            hard_stop("Protected field contract violation", {"split": split, "headers": headers})
        for row_id, row in enumerate(reader):
            raw_title = row.get("title")
            raw_label = row.get("label")
            label = canonicalize_label_p0(raw_label)
            quality = classify_title(raw_title)
            norm = normalize_text_p0(raw_title)
            if label not in EXPECTED_BANGLABAIT_LABEL_DOMAIN:
                hard_stop("Protected label outside frozen {0,1} domain", {"split": split, "row_id": row_id, "raw_label": raw_label})
            if quality in {"MISSING","EMPTY","KNOWN_PLACEHOLDER","NON_TEXTUAL_GARBAGE"}:
                hard_stop("Malformed protected title encountered although Stage-02 reported only overlap blockers",
                          {"split": split, "row_id": row_id, "quality": quality})
            records.append({
                "frozen_package_sha256": EXPECTED_ZIP_SHA256,
                "source_artifact": artifact_id,
                "source_row_id": int(row_id),
                "original_split": split,
                "title": raw_title,
                "label": int(label),
                "raw_title_sha256": sha_text(raw_title),
                "normalized_title": norm,
                "normalized_title_sha256": sha_text(norm),
                "title_quality": quality,
            })
    if not records:
        hard_stop("Protected split has zero rows", {"split": split})
    return records

ARTIFACT_IDS = {"train":"banglabait_train", "dev":"banglabait_dev", "test":"banglabait_test"}
rows_by_split = {s: load_protected_split(SOURCE_PATHS[s], s, ARTIFACT_IDS[s]) for s in ["train","dev","test"]}


def index_by(rows, key):
    d = defaultdict(list)
    for r in rows:
        d[r[key]].append(r)
    return d


def pair_forensics(left_name, right_name, sample_limit=10):
    left, right = rows_by_split[left_name], rows_by_split[right_name]
    lraw, rraw = index_by(left,"raw_title_sha256"), index_by(right,"raw_title_sha256")
    lnorm, rnorm = index_by(left,"normalized_title_sha256"), index_by(right,"normalized_title_sha256")
    exact_shared = sorted(set(lraw) & set(rraw))
    norm_shared = sorted(set(lnorm) & set(rnorm))
    exact_norm = set()
    for rh in exact_shared:
        for rr in lraw[rh]: exact_norm.add(rr["normalized_title_sha256"])
        for rr in rraw[rh]: exact_norm.add(rr["normalized_title_sha256"])
    normalization_only_shared = sorted(set(norm_shared) - exact_norm)
    exact_left_ids = {r["source_row_id"] for h in exact_shared for r in lraw[h]}
    exact_right_ids = {r["source_row_id"] for h in exact_shared for r in rraw[h]}
    norm_left_ids = {r["source_row_id"] for h in norm_shared for r in lnorm[h]}
    norm_right_ids = {r["source_row_id"] for h in norm_shared for r in rnorm[h]}
    normonly_left_ids = set()
    normonly_right_ids = set()
    right_raw_set = set(rraw); left_raw_set = set(lraw)
    for h in norm_shared:
        for r in lnorm[h]:
            if r["raw_title_sha256"] not in right_raw_set: normonly_left_ids.add(r["source_row_id"])
        for r in rnorm[h]:
            if r["raw_title_sha256"] not in left_raw_set: normonly_right_ids.add(r["source_row_id"])
    conflict_hashes = []
    label_pairs = Counter()
    for h in norm_shared:
        ll = sorted({r["label"] for r in lnorm[h]}); rr = sorted({r["label"] for r in rnorm[h]})
        for a in ll:
            for b in rr:
                label_pairs[(str(a),str(b))] += 1
        if any(a != b for a in ll for b in rr): conflict_hashes.append(h)
    examples = []
    for h in norm_shared[:sample_limit]:
        examples.append({
            "normalized_sha256": h,
            "exact_raw_overlap_present": any(r["raw_title_sha256"] in set(rraw) for r in lnorm[h]),
            "left": [{k:r[k] for k in ["source_row_id","raw_title_sha256","label","title_quality","title"]} for r in lnorm[h][:3]],
            "right": [{k:r[k] for k in ["source_row_id","raw_title_sha256","label","title_quality","title"]} for r in rnorm[h][:3]],
        })
    return {
        "left_split": left_name, "right_split": right_name,
        "exact_raw_shared_hashes": len(exact_shared),
        "normalized_shared_hashes": len(norm_shared),
        "normalization_only_shared_hashes": len(normalization_only_shared),
        "exact_raw_affected_rows_left": len(exact_left_ids),
        "exact_raw_affected_rows_right": len(exact_right_ids),
        "normalized_affected_rows_left": len(norm_left_ids),
        "normalized_affected_rows_right": len(norm_right_ids),
        "normalization_only_affected_rows_left": len(normonly_left_ids),
        "normalization_only_affected_rows_right": len(normonly_right_ids),
        "conflicting_label_shared_hashes": len(conflict_hashes),
        "label_pair_shared_hash_counts": [
            {"left_label":a,"right_label":b,"shared_normalized_hashes":int(n)}
            for (a,b),n in sorted(label_pairs.items())],
        "representative_samples": examples,
    }

pair_reports = [pair_forensics("train","dev"), pair_forensics("train","test"), pair_forensics("dev","test")]

# Within-split normalized-title conflicting-label classes.
within_conflicts = {}
for split, rows in rows_by_split.items():
    idx = index_by(rows,"normalized_title_sha256")
    groups = []
    for h, grp in sorted(idx.items()):
        labels = sorted({r["label"] for r in grp})
        if len(labels) > 1:
            groups.append({"normalized_sha256":h, "row_ids":sorted(r["source_row_id"] for r in grp),
                           "labels":labels, "representative_titles":[r["title"] for r in grp[:4]]})
    within_conflicts[split] = {"group_count":len(groups), "affected_rows":len({rid for g in groups for rid in g["row_ids"]}), "groups":groups}

# Frozen complete-equivalence-class quarantine rule.
norm_to_splits = defaultdict(set)
for split, rows in rows_by_split.items():
    for r in rows: norm_to_splits[r["normalized_title_sha256"]].add(split)
cross_split_hashes = {h for h, ss in norm_to_splits.items() if len(ss) > 1}
within_conflict_hashes = {g["normalized_sha256"] for v in within_conflicts.values() for g in v["groups"]}
quarantine_hashes = cross_split_hashes | within_conflict_hashes

manifest_records = []
clean_rows_by_split = {}
for split, rows in rows_by_split.items():
    clean = []
    for r in rows:
        reasons = []
        h = r["normalized_title_sha256"]
        if h in cross_split_hashes: reasons.append("CROSS_SPLIT_NORMALIZED_TITLE_CLASS")
        if h in within_conflict_hashes: reasons.append("WITHIN_SPLIT_CONFLICTING_LABEL_CLASS")
        decision = "QUARANTINE" if reasons else "KEEP"
        manifest_records.append({
            "frozen_package_sha256": EXPECTED_ZIP_SHA256,
            "source_artifact": r["source_artifact"], "source_row_id": r["source_row_id"],
            "raw_title_sha256": r["raw_title_sha256"], "normalized_title_sha256": h,
            "original_split": split, "decision": decision,
            "reason_code": "|".join(reasons) if reasons else "KEEP",
            "decontamination_rule_version": DECONTAM_VERSION,
        })
        if decision == "KEEP": clean.append(r)
    clean_rows_by_split[split] = clean

manifest_records = sorted(manifest_records, key=lambda x:(x["original_split"],x["source_row_id"],x["normalized_title_sha256"]))
derived_manifest_sha = sha256_bytes(canonical_json_bytes(manifest_records))
for r in manifest_records:
    r["derived_manifest_sha256"] = derived_manifest_sha

clean_dir = OUT_ROOT / "clean_splits"; clean_dir.mkdir(parents=True, exist_ok=True)
clean_paths = {}
for split, rows in clean_rows_by_split.items():
    df = pd.DataFrame([{
        "source_artifact":r["source_artifact"], "source_row_id":r["source_row_id"],
        "title":r["title"], "label":r["label"], "raw_title_sha256":r["raw_title_sha256"],
        "normalized_title_sha256":r["normalized_title_sha256"], "original_split":split
    } for r in rows]).sort_values("source_row_id", kind="stable").reset_index(drop=True)
    p = clean_dir / f"banglabait_{split}_clean.parquet"
    df.to_parquet(p, index=False)
    clean_paths[split] = p

pd.DataFrame(manifest_records).to_parquet(OUT_ROOT/"CLEAN_ROW_MANIFEST.parquet", index=False)
write_json(OUT_ROOT/"CLEAN_DATA_MANIFEST.json", {
    "schema_version":1, "protocol_version":PROTOCOL_VERSION, "decontamination_rule_version":DECONTAM_VERSION,
    "frozen_package_sha256":EXPECTED_ZIP_SHA256, "derived_manifest_sha256":derived_manifest_sha,
    "record_count":len(manifest_records), "records":manifest_records,
    "clean_split_paths":{s:str(p.relative_to(OUT_ROOT)) for s,p in clean_paths.items()},
})

forensics = {
    "schema_version":1, "p0_methodology":P0_METHOD, "decontamination_rule_version":DECONTAM_VERSION,
    "pairwise_protected_title_forensics":pair_reports,
    "within_split_conflicts":within_conflicts,
    "cross_split_normalized_equivalence_classes":len(cross_split_hashes),
    "within_split_conflicting_equivalence_classes":len(within_conflict_hashes),
    "total_quarantined_equivalence_classes":len(quarantine_hashes),
    "quarantined_rows_by_split":{s:len(rows_by_split[s])-len(clean_rows_by_split[s]) for s in rows_by_split},
}
write_json(OUT_ROOT/"DECONTAMINATION_FORENSICS.json", forensics)
print(json.dumps({"quarantined_rows":forensics["quarantined_rows_by_split"], "cross_split_classes":len(cross_split_hashes)}, indent=2))


{
  "quarantined_rows": {
    "train": 295,
    "dev": 105,
    "test": 228
  },
  "cross_split_classes": 290
}


In [5]:
# ---------- D0 hard gate ----------

def clean_overlap_count(a,b):
    return len({r["normalized_title_sha256"] for r in clean_rows_by_split[a]} &
               {r["normalized_title_sha256"] for r in clean_rows_by_split[b]})


def conflict_count(rows):
    idx = index_by(rows,"normalized_title_sha256")
    return sum(len({r["label"] for r in grp}) > 1 for grp in idx.values())

summary_rows = []
gate_checks = []
for split in ["train","dev","test"]:
    orig = pd.DataFrame(rows_by_split[split]); clean = pd.DataFrame(clean_rows_by_split[split])
    orig_n, clean_n = len(orig), len(clean)
    orig_prev = float(orig["label"].mean()); clean_prev = float(clean["label"].mean()) if clean_n else float("nan")
    split_retention = clean_n/orig_n
    summary_rows.append({"split":split,"original_rows":orig_n,"clean_rows":clean_n,"quarantined_rows":orig_n-clean_n,
                         "row_retention":split_retention,"original_positive_prevalence":orig_prev,
                         "clean_positive_prevalence":clean_prev,"prevalence_change_abs_pp":abs(clean_prev-orig_prev)*100})
    gate_checks.append({"check":f"{split}_row_retention_ge_0.80","pass":split_retention>=0.80,"value":split_retention})
    gate_checks.append({"check":f"{split}_prevalence_change_le_5pp","pass":abs(clean_prev-orig_prev)<=0.05,"value":abs(clean_prev-orig_prev)})
    class_min = {"train":2000,"dev":250,"test":500}[split]
    for label in [0,1]:
        on = int((orig["label"]==label).sum()); cn = int((clean["label"]==label).sum())
        retention = cn/on if on else 0.0
        summary_rows.append({"split":split,"class_label":label,"original_class_rows":on,"clean_class_rows":cn,"class_retention":retention})
        gate_checks.append({"check":f"{split}_class_{label}_retention_ge_0.80","pass":retention>=0.80,"value":retention})
        gate_checks.append({"check":f"{split}_class_{label}_count_ge_{class_min}","pass":cn>=class_min,"value":cn})

for a,b in [("train","dev"),("train","test"),("dev","test")]:
    n=clean_overlap_count(a,b)
    gate_checks.append({"check":f"clean_{a}_{b}_normalized_overlap_zero","pass":n==0,"value":n})
for split in ["train","dev","test"]:
    n=conflict_count(clean_rows_by_split[split])
    gate_checks.append({"check":f"clean_{split}_conflicting_title_groups_zero","pass":n==0,"value":n})

pd.DataFrame(summary_rows).to_csv(OUT_ROOT/"CLEAN_SPLIT_SUMMARY.csv", index=False)
D0_PASS = all(x["pass"] for x in gate_checks)
D0_REPORT = {
    "schema_version":1, "gate":"D0", "status":"PASS" if D0_PASS else "HARD_STOP",
    "decontamination_rule_version":DECONTAM_VERSION, "derived_manifest_sha256":derived_manifest_sha,
    "checks":gate_checks, "clean_split_summary_csv":"CLEAN_SPLIT_SUMMARY.csv",
    "scientific_firewall_note":"DEV/TEST labels were read only for D0 integrity/support auditing; no DEV/TEST value may enter model development.",
}
write_json(OUT_ROOT/"D0_REPORT.json", D0_REPORT)
if not D0_PASS:
    hard_stop("D0 residual-support/zero-leakage gate failed. Model fitting is forbidden.", D0_REPORT)
print("D0: PASS — Stage-03 TRAIN-only model development authorized")


D0: PASS — Stage-03 TRAIN-only model development authorized


## Cue Schema and TRAIN-only Development Split
The eight locked cue families and four structural controls are frozen before external evaluation. Configuration development uses only the deterministic 90/10 split of CLEAN TRAIN grouped by normalized-title hash.


In [6]:
# ---------- Frozen cue schema BANGLA-CLICKBAIT-CUES-v1.0.0 ----------
CUE_SCHEMA = {
    "schema_version":1,
    "cue_version":CUE_VERSION,
    "normalization":"P0-v1.1.1 NFKC + remove U+200B/U+200C/U+200D/U+FEFF + casefold + trim + collapse whitespace; punctuation retained",
    "families":{
        "curiosity_information_withholding":{
            "type":"lexical_phrase_count",
            "terms":["রহস্য","গোপন","ফাঁস","অজানা","আসল কারণ","শেষ পর্যন্ত","অবশেষে","কি ঘটল","কী ঘটল","কি হলো","কী হলো","জানলে","চমক"]},
        "interrogative_structure":{
            "type":"lexical_syntactic_count",
            "terms":["কি","কী","কে","কেন","কবে","কোথায়","কোথায়","কিভাবে","কীভাবে","কোন","কত","কার","কাকে","কারা"],
            "punctuation":["?"]},
        "vague_deictic_reference":{
            "type":"lexical_referential_count",
            "terms":["এই","ওই","সেই","এটা","ওটা","সেটা","এমন","তেমন","এভাবে","ওভাবে","সেভাবে","যা","যেটা"]},
        "sensational_emotional_loading":{
            "type":"lexical_count",
            "terms":["চাঞ্চল্যকর","অবিশ্বাস্য","ভয়ংকর","ভয়ংকর","বিস্ময়কর","বিস্ময়কর","হতবাক","তোলপাড়","তোলপাড়","হইচই","ভাইরাল","শিউরে","মর্মান্তিক","উত্তাল"]},
        "exaggeration_superlative":{
            "type":"lexical_count",
            "terms":["সবচেয়ে","সবচেয়ে","সেরা","একমাত্র","সর্বোচ্চ","সর্বনিম্ন","বিশ্বসেরা","ইতিহাসে","জীবনে","কখনও","কখনো","শতভাগ","নিশ্চিত","অসাধারণ"]},
        "specificity_numerals":{
            "type":"surface_count",
            "patterns":[r"[0-9০-৯]+", r"[0-9০-৯]+(?:[.,][0-9০-৯]+)?%", r"(?:৳|টাকা|tk|bdt)\s*[0-9০-৯]+"]},
        "punctuation_intensity":{
            "type":"surface_count",
            "punctuation":["!","?","…"],
            "repeated_pattern":r"[!?…]{2,}"},
        "direct_address_imperative":{
            "type":"lexical_syntactic_count",
            "terms":["আপনি","আপনার","আপনাকে","তুমি","তোমার","তোমাকে","দেখুন","জানুন","জেনে নিন","দেখে নিন","পড়ুন","পড়ুন","শুনুন","ভাবুন","বলুন"]},
    },
    "structural_controls":["token_count","character_count","mean_token_length","latin_code_mixing_proportion"],
    "perturbable_families":["curiosity_information_withholding","direct_address_imperative","punctuation_intensity"],
}
CUE_SCHEMA["cue_schema_sha256"] = sha256_bytes(canonical_json_bytes({k:v for k,v in CUE_SCHEMA.items() if k!="cue_schema_sha256"}))
write_json(OUT_ROOT/"CUE_FEATURE_SCHEMA.json", CUE_SCHEMA)


def _term_count(text, terms):
    # Token-boundary behavior is deterministic: spaces delimit lexical terms; multiword phrases use literal normalized substring.
    tokens = text.split()
    token_counts = Counter(tokens)
    total = 0
    for term in terms:
        t = normalize_text_p0(term)
        total += text.count(t) if " " in t else token_counts.get(t,0)
    return int(total)


def extract_cue_features(titles, schema=CUE_SCHEMA):
    out=[]
    for raw in titles:
        text=normalize_text_p0(raw); toks=text.split()
        row={}
        for name,spec in schema["families"].items():
            if name=="specificity_numerals":
                c=sum(len(re.findall(p,text,flags=re.I)) for p in spec["patterns"])
            elif name=="punctuation_intensity":
                c=sum(text.count(p) for p in spec["punctuation"]) + len(re.findall(spec["repeated_pattern"],text))
            else:
                c=_term_count(text,spec.get("terms",[])) + sum(text.count(p) for p in spec.get("punctuation",[]))
            row[name]=int(c)
        chars=len(text); latin=sum(("a"<=ch<="z") for ch in text)
        row.update({
            "token_count":len(toks), "character_count":chars,
            "mean_token_length":float(np.mean([len(t) for t in toks])) if toks else 0.0,
            "latin_code_mixing_proportion":latin/chars if chars else 0.0,
        })
        out.append(row)
    cols=list(schema["families"].keys())+schema["structural_controls"]
    return pd.DataFrame(out,columns=cols)

CUE_COLUMNS=list(CUE_SCHEMA["families"].keys())+CUE_SCHEMA["structural_controls"]
print("Cue schema locked:", CUE_SCHEMA["cue_schema_sha256"])


Cue schema locked: 200e3ccee3dd6e11b57728b8d6d5ac91bd88cc2e8c3c5e8986d9daa145d73556


In [7]:
# ---------- Deterministic TRAIN-only 90/10 stratified group development split ----------
clean_train = pd.read_parquet(clean_paths["train"]).sort_values("source_row_id",kind="stable").reset_index(drop=True)
sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=CONFIG_SPLIT_SEED)
splits=list(sgkf.split(clean_train["title"], clean_train["label"], groups=clean_train["normalized_title_sha256"]))
internal_train_idx, internal_dev_idx = splits[0]
if set(clean_train.iloc[internal_train_idx]["normalized_title_sha256"]) & set(clean_train.iloc[internal_dev_idx]["normalized_title_sha256"]):
    hard_stop("TRAIN-only internal group split leaked normalized-title groups")

internal_assign = pd.DataFrame({
    "source_row_id":clean_train["source_row_id"],
    "normalized_title_sha256":clean_train["normalized_title_sha256"],
    "label":clean_train["label"],
    "assignment":"internal_train",
})
internal_assign.loc[internal_dev_idx,"assignment"]="internal_dev"
internal_assign=internal_assign.sort_values("source_row_id",kind="stable")
internal_assign.to_parquet(OUT_ROOT/"TRAIN_DEV_SPLIT_ROWS.parquet",index=False)
split_manifest={
    "schema_version":1,"split_seed":CONFIG_SPLIT_SEED,"method":"StratifiedGroupKFold(n_splits=10, shuffle=True), fold_index=0",
    "group":"normalized_title_sha256","development_ratio_target":"90/10",
    "internal_train_rows":int(len(internal_train_idx)),"internal_dev_rows":int(len(internal_dev_idx)),
    "internal_train_positive_prevalence":float(clean_train.iloc[internal_train_idx]["label"].mean()),
    "internal_dev_positive_prevalence":float(clean_train.iloc[internal_dev_idx]["label"].mean()),
    "rows_manifest":"TRAIN_DEV_SPLIT_ROWS.parquet",
    "rows_manifest_sha256":sha256_file(OUT_ROOT/"TRAIN_DEV_SPLIT_ROWS.parquet"),
    "dev_test_firewall":"No BanglaBait DEV/TEST observation was used to construct this split.",
}
write_json(OUT_ROOT/"TRAIN_DEV_SPLIT_MANIFEST.json",split_manifest)

Xtr=clean_train.iloc[internal_train_idx].reset_index(drop=True)
Xdv=clean_train.iloc[internal_dev_idx].reset_index(drop=True)
print("TRAIN-only internal split:",len(Xtr),len(Xdv))


TRAIN-only internal split: 9486 1058


## Model Development Procedure
The locked model panel is EBM (`interactions=0`), TF-IDF + L2 Logistic Regression, BanglaBERT, and XLM-R. The five final refit seeds are preserved exactly. DEV, TEST, and external data do not enter model selection.


In [8]:
# ---------- Source-only configuration development contracts ----------
from interpret.glassbox import ExplainableBoostingClassifier
from huggingface_hub import model_info
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

if not torch.cuda.is_available():
    hard_stop(
        "CUDA GPU is unavailable for the complete Stage-03 transformer experiment",
        {"why":"BanglaBERT and XLM-R training are mandatory in the locked model panel.",
         "safe_to_continue":False,
         "action":"Enable a Kaggle GPU accelerator and rerun Stage 03 from the top."}
    )
print("GPU:", torch.cuda.get_device_name(0))


def seed_everything(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    try: torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception: pass


def supported_kwargs(callable_obj, kwargs):
    sig=inspect.signature(callable_obj)
    return {k:v for k,v in kwargs.items() if k in sig.parameters}

# Resolve mutable HF names to immutable repository revisions BEFORE any model fitting.
HF_REVISIONS={}
for key,repo in HF_MODELS.items():
    try:
        info=model_info(repo)
        if not info.sha or not re.fullmatch(r"[0-9a-f]{40}",info.sha):
            hard_stop("Could not resolve immutable Hugging Face revision",{"repo":repo,"sha":info.sha})
        HF_REVISIONS[key]=info.sha
    except Exception as e:
        hard_stop("Hugging Face model identity resolution failed; Stage 03 requires network access to resolve the locked model name to an immutable commit",
                  {"repo":repo,"error":f"{type(e).__name__}: {e}"})

EBM_CANDIDATES=[
    {"max_bins":32,"learning_rate":0.05,"max_rounds":1000},
    {"max_bins":64,"learning_rate":0.05,"max_rounds":1000},
    {"max_bins":64,"learning_rate":0.03,"max_rounds":1500},
]
LR_CANDIDATES=[0.25,1.0,4.0]
TRANSFORMER_BASE_CONFIG={"learning_rate":2e-5,"weight_decay":0.01,"batch_size":16,"max_length":128,"max_epochs":4,"warmup_ratio":0.1}
CONFIG_SELECTION_METRIC="internal_train_split_AUROC"
print("HF immutable revisions:",HF_REVISIONS)


GPU: Tesla T4
HF immutable revisions: {'banglabert': '9ce791f330578f50da6bc52b54205166fb5d1c8c', 'xlmr': 'e73636d4f797dec63c3081bb6ed5c7b0bb3f2089'}


In [9]:
# ---------- EBM and TF-IDF/LR configuration development on CLEAN TRAIN only ----------
Xtr_cues=extract_cue_features(Xtr["title"].tolist()).astype(float)
Xdv_cues=extract_cue_features(Xdv["title"].tolist()).astype(float)
ytr=Xtr["label"].to_numpy(); ydv=Xdv["label"].to_numpy()

config_results={"ebm":[],"tfidf_lr":[],"banglabert":[],"xlmr":[]}

for i,cand in enumerate(EBM_CANDIDATES):
    seed_everything(CONFIG_SPLIT_SEED)
    base={"interactions":0,"random_state":CONFIG_SPLIT_SEED,"validation_size":0,**cand}
    model=ExplainableBoostingClassifier(**supported_kwargs(ExplainableBoostingClassifier,base))
    model.fit(Xtr_cues,ytr)
    p=model.predict_proba(Xdv_cues)[:,1]
    auc=float(roc_auc_score(ydv,p))
    config_results["ebm"].append({"candidate_index":i,"params":base,"internal_dev_auroc":auc})

for i,C in enumerate(LR_CANDIDATES):
    seed_everything(CONFIG_SPLIT_SEED)
    word=TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=100000,sublinear_tf=True,dtype=np.float32)
    char=TfidfVectorizer(analyzer="char",ngram_range=(3,5),min_df=2,max_features=100000,sublinear_tf=True,dtype=np.float32)
    pipe=Pipeline([("features",FeatureUnion([("word",word),("char",char)])),
                   ("clf",LogisticRegression(C=C,solver="liblinear",max_iter=2000,random_state=CONFIG_SPLIT_SEED))])
    pipe.fit(Xtr["title"],ytr)
    p=expit(pipe.decision_function(Xdv["title"]))
    auc=float(roc_auc_score(ydv,p))
    config_results["tfidf_lr"].append({"candidate_index":i,"params":{"C":C},"internal_dev_auroc":auc})


def select_best(rows):
    # Highest AUROC; exact ties choose the smallest candidate/epoch index.
    return sorted(rows,key=lambda r:(-r["internal_dev_auroc"],r.get("candidate_index",r.get("epoch",999))))[0]

selected_ebm=select_best(config_results["ebm"])
selected_lr=select_best(config_results["tfidf_lr"])
print("Selected EBM:",selected_ebm)
print("Selected LR:",selected_lr)


/usr/local/lib/python3.12/dist-packages/interpret/glassbox/_ebm/_ebm.py:555: UserWarning: If validation_size is 0, the outer_bags have no purpose. Set outer_bags=1 to remove this warning.
  warn(
/usr/local/lib/python3.12/dist-packages/interpret/glassbox/_ebm/_ebm.py:555: UserWarning: If validation_size is 0, the outer_bags have no purpose. Set outer_bags=1 to remove this warning.
  warn(
/usr/local/lib/python3.12/dist-packages/interpret/glassbox/_ebm/_ebm.py:555: UserWarning: If validation_size is 0, the outer_bags have no purpose. Set outer_bags=1 to remove this warning.
  warn(


Selected EBM: {'candidate_index': 0, 'params': {'interactions': 0, 'random_state': 20260906, 'validation_size': 0, 'max_bins': 32, 'learning_rate': 0.05, 'max_rounds': 1000}, 'internal_dev_auroc': 0.733208800332088}
Selected LR: {'candidate_index': 1, 'params': {'C': 1.0}, 'internal_dev_auroc': 0.8004546442902606}


In [10]:
# ---------- Transformer TRAIN-only epoch selection ----------
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.enc=tokenizer(list(texts),truncation=True,padding=False,max_length=max_length)
        self.labels=[int(x) for x in labels]
    def __len__(self): return len(self.labels)
    def __getitem__(self,i):
        d={k:torch.tensor(v[i],dtype=torch.long) for k,v in self.enc.items()}
        d["labels"]=torch.tensor(self.labels[i],dtype=torch.long); return d


def collate_dynamic(batch, tokenizer):
    labels=torch.stack([x.pop("labels") for x in batch])
    padded=tokenizer.pad(batch,padding=True,return_tensors="pt")
    padded["labels"]=labels
    return padded


def eval_logits(model,loader,device):
    model.eval(); logits=[]; ys=[]
    with torch.no_grad():
        for batch in loader:
            y=batch.pop("labels").numpy(); ys.extend(y.tolist())
            batch={k:v.to(device) for k,v in batch.items()}
            out=model(**batch).logits.detach().cpu().numpy()
            logits.extend((out[:,1]-out[:,0]).tolist())
    return np.asarray(ys,int),np.asarray(logits,float)


def develop_transformer(model_key):
    repo=HF_MODELS[model_key]; rev=HF_REVISIONS[model_key]; cfg=TRANSFORMER_BASE_CONFIG
    seed_everything(CONFIG_SPLIT_SEED)
    tok=AutoTokenizer.from_pretrained(repo,revision=rev,use_fast=True)
    trds=TextDataset(Xtr["title"],Xtr["label"],tok,cfg["max_length"])
    dvds=TextDataset(Xdv["title"],Xdv["label"],tok,cfg["max_length"])
    trloader=DataLoader(trds,batch_size=cfg["batch_size"],shuffle=True,
                        generator=torch.Generator().manual_seed(CONFIG_SPLIT_SEED),
                        collate_fn=lambda b:collate_dynamic(b,tok))
    dvloader=DataLoader(dvds,batch_size=max(32,cfg["batch_size"]*2),shuffle=False,
                        collate_fn=lambda b:collate_dynamic(b,tok))
    model=AutoModelForSequenceClassification.from_pretrained(repo,revision=rev,num_labels=2)
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu"); model.to(device)
    optimizer=torch.optim.AdamW(model.parameters(),lr=cfg["learning_rate"],weight_decay=cfg["weight_decay"])
    total_steps=len(trloader)*cfg["max_epochs"]; warmup=int(round(total_steps*cfg["warmup_ratio"]))
    scheduler=get_linear_schedule_with_warmup(optimizer,warmup,total_steps)
    rows=[]
    for epoch in range(1,cfg["max_epochs"]+1):
        model.train()
        for batch in trloader:
            batch={k:v.to(device) for k,v in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            try:
                loss=model(**batch).loss
                loss.backward()
            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    torch.cuda.empty_cache()
                    hard_stop("CUDA out of memory during transformer configuration development",
                              {"model":model_key,"batch_size":cfg["batch_size"],
                               "safe_to_continue":False,
                               "action":"Use a Kaggle GPU runtime with sufficient memory; do not reduce data, seeds, or the locked model panel."})
                raise
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            optimizer.step(); scheduler.step()
        yy,zz=eval_logits(model,dvloader,device)
        auc=float(roc_auc_score(yy,zz))
        rows.append({"epoch":epoch,"internal_dev_auroc":auc})
        print(model_key,"epoch",epoch,"AUROC",auc)
    del model; torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return rows

for key in ["banglabert","xlmr"]:
    config_results[key]=develop_transformer(key)

selected_transformers={key:select_best(config_results[key]) for key in ["banglabert","xlmr"]}
print("Selected transformer epochs:",selected_transformers)


config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/443M [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


xlmr epoch 1 AUROC 0.839055921247702
xlmr epoch 2 AUROC 0.8625432406254324
xlmr epoch 3 AUROC 0.8738895807388959
xlmr epoch 4 AUROC 0.8712131095692739
Selected transformer epochs: {'banglabert': {'epoch': 2, 'internal_dev_auroc': 0.8695763901243353}, 'xlmr': {'epoch': 3, 'internal_dev_auroc': 0.8738895807388959}}


In [11]:
# ---------- Freeze MODEL_CONFIG_LOCK before final five-seed refits ----------
MODEL_CONFIG_LOCK={
    "schema_version":1,"protocol_version":PROTOCOL_VERSION,"selection_metric":CONFIG_SELECTION_METRIC,
    "configuration_data_boundary":"CLEAN BanglaBait TRAIN internal 90/10 split only",
    "configuration_split_manifest_sha256":sha256_file(OUT_ROOT/"TRAIN_DEV_SPLIT_MANIFEST.json"),
    "models":{
        "ebm":{"class":"interpret.glassbox.ExplainableBoostingClassifier","interactions":0,
               "selected":selected_ebm,"features":CUE_COLUMNS,"cue_schema_sha256":CUE_SCHEMA["cue_schema_sha256"]},
        "tfidf_lr":{"class":"TF-IDF FeatureUnion + L2 LogisticRegression","word_ngrams":[1,2],"char_ngrams":[3,5],
                    "selected":selected_lr,"min_df":2,"max_features_per_branch":100000,"sublinear_tf":True,"solver":"liblinear"},
        "banglabert":{"repo_id":HF_MODELS["banglabert"],"revision":HF_REVISIONS["banglabert"],
                       "base_config":TRANSFORMER_BASE_CONFIG,"selected_epoch":selected_transformers["banglabert"]["epoch"]},
        "xlmr":{"repo_id":HF_MODELS["xlmr"],"revision":HF_REVISIONS["xlmr"],
                "base_config":TRANSFORMER_BASE_CONFIG,"selected_epoch":selected_transformers["xlmr"]["epoch"]},
    },
    "candidate_results":config_results,
    "final_refit_seeds":SEEDS,
    "firewall":{"DEV":"calibration only Stage 04","TEST":"locked evaluation Stage 05","external":"Stage 05 inference only"},
}
MODEL_CONFIG_LOCK["config_lock_sha256"]=sha256_bytes(canonical_json_bytes({k:v for k,v in MODEL_CONFIG_LOCK.items() if k!="config_lock_sha256"}))
write_json(OUT_ROOT/"MODEL_CONFIG_LOCK.json",MODEL_CONFIG_LOCK)
print("MODEL_CONFIG_LOCK:",MODEL_CONFIG_LOCK["config_lock_sha256"])


MODEL_CONFIG_LOCK: af6f73a7b839c70f2d9f592b6091f35c8ad24c5e8af5ae227812d6452128fa0a


In [12]:
# ---------- Final five-seed full CLEAN TRAIN refits: EBM + TF-IDF/LR ----------
models_dir=OUT_ROOT/"models"; models_dir.mkdir(parents=True,exist_ok=True)
full_cues=extract_cue_features(clean_train["title"].tolist()).astype(float)
yfull=clean_train["label"].to_numpy()

for seed in SEEDS:
    seed_everything(seed)
    ebm_params={**selected_ebm["params"],"interactions":0,"random_state":seed,"validation_size":0}
    ebm=ExplainableBoostingClassifier(**supported_kwargs(ExplainableBoostingClassifier,ebm_params))
    ebm.fit(full_cues,yfull)
    ep=models_dir/f"ebm_seed{seed}.joblib"; joblib.dump(ebm,ep,compress=3)

    seed_everything(seed)
    C=float(selected_lr["params"]["C"])
    word=TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=100000,sublinear_tf=True,dtype=np.float32)
    char=TfidfVectorizer(analyzer="char",ngram_range=(3,5),min_df=2,max_features=100000,sublinear_tf=True,dtype=np.float32)
    lr=Pipeline([("features",FeatureUnion([("word",word),("char",char)])),
                 ("clf",LogisticRegression(C=C,solver="liblinear",max_iter=2000,random_state=seed))])
    lr.fit(clean_train["title"],yfull)
    lp=models_dir/f"tfidf_lr_seed{seed}.joblib"; joblib.dump(lr,lp,compress=3)
print("Classical final refits complete")


/usr/local/lib/python3.12/dist-packages/interpret/glassbox/_ebm/_ebm.py:555: UserWarning: If validation_size is 0, the outer_bags have no purpose. Set outer_bags=1 to remove this warning.
  warn(
/usr/local/lib/python3.12/dist-packages/interpret/glassbox/_ebm/_ebm.py:555: UserWarning: If validation_size is 0, the outer_bags have no purpose. Set outer_bags=1 to remove this warning.
  warn(
/usr/local/lib/python3.12/dist-packages/interpret/glassbox/_ebm/_ebm.py:555: UserWarning: If validation_size is 0, the outer_bags have no purpose. Set outer_bags=1 to remove this warning.
  warn(
/usr/local/lib/python3.12/dist-packages/interpret/glassbox/_ebm/_ebm.py:555: UserWarning: If validation_size is 0, the outer_bags have no purpose. Set outer_bags=1 to remove this warning.
  warn(
/usr/local/lib/python3.12/dist-packages/interpret/glassbox/_ebm/_ebm.py:555: UserWarning: If validation_size is 0, the outer_bags have no purpose. Set outer_bags=1 to remove this warning.
  warn(


Classical final refits complete


In [13]:
# ---------- Final five-seed full CLEAN TRAIN refits: BanglaBERT + XLM-R ----------
def train_full_transformer(model_key,seed):
    repo=HF_MODELS[model_key]; rev=HF_REVISIONS[model_key]
    cfg=TRANSFORMER_BASE_CONFIG; epochs=int(MODEL_CONFIG_LOCK["models"][model_key]["selected_epoch"])
    seed_everything(seed)
    tok=AutoTokenizer.from_pretrained(repo,revision=rev,use_fast=True)
    ds=TextDataset(clean_train["title"],clean_train["label"],tok,cfg["max_length"])
    loader=DataLoader(ds,batch_size=cfg["batch_size"],shuffle=True,
                      generator=torch.Generator().manual_seed(seed),collate_fn=lambda b:collate_dynamic(b,tok))
    model=AutoModelForSequenceClassification.from_pretrained(repo,revision=rev,num_labels=2)
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu"); model.to(device)
    opt=torch.optim.AdamW(model.parameters(),lr=cfg["learning_rate"],weight_decay=cfg["weight_decay"])
    total=len(loader)*epochs; warmup=int(round(total*cfg["warmup_ratio"]))
    sched=get_linear_schedule_with_warmup(opt,warmup,total)
    for epoch in range(epochs):
        model.train()
        for batch in loader:
            batch={k:v.to(device) for k,v in batch.items()}; opt.zero_grad(set_to_none=True)
            try:
                loss=model(**batch).loss
                loss.backward()
            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    torch.cuda.empty_cache()
                    hard_stop("CUDA out of memory during final transformer refit",
                              {"model":model_key,"seed":seed,"batch_size":cfg["batch_size"],
                               "safe_to_continue":False,
                               "action":"Use a Kaggle GPU runtime with sufficient memory; do not reduce data, seeds, or the locked model panel."})
                raise
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            opt.step(); sched.step()
    target=models_dir/f"{model_key}_seed{seed}"; target.mkdir(parents=True,exist_ok=False)
    model.save_pretrained(target,safe_serialization=True)
    tok.save_pretrained(target)
    del model; torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return target

for key in ["banglabert","xlmr"]:
    for seed in SEEDS:
        print("Final refit",key,seed)
        train_full_transformer(key,seed)
print("Transformer final refits complete")


Final refit banglabert 42


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraForSequenceClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.weight            | UNEXPECTED | 
electra.embeddings.position_ids                   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoi

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final refit banglabert 137


...

Final refit xlmr 65537


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Transformer final refits complete


## Output Artifacts and Stage Decision
All required artifacts are hashed and checked before a final `PASS`. Scientific gate failures and unsafe runtime failures emit `HARD_STOP`; they are never converted to partial success.


In [14]:
# ---------- Immutable artifact manifest, metadata, output hashes ----------
def recursive_file_manifest(path):
    path=Path(path)
    if path.is_file():
        return [{"path":str(path.relative_to(OUT_ROOT)).replace("\\","/"),"size":path.stat().st_size,"sha256":sha256_file(path)}]
    return [{"path":str(p.relative_to(OUT_ROOT)).replace("\\","/"),"size":p.stat().st_size,"sha256":sha256_file(p)}
            for p in sorted(path.rglob("*"),key=str) if p.is_file()]

model_records=[]
for key in MODEL_KEYS:
    for seed in SEEDS:
        p=models_dir/(f"{key}_seed{seed}.joblib" if key in {"ebm","tfidf_lr"} else f"{key}_seed{seed}")
        if not p.exists(): hard_stop("Expected final model artifact missing",{"model":key,"seed":seed,"path":str(p)})
        files=recursive_file_manifest(p)
        state_hash=sha256_bytes(canonical_json_bytes(files))
        model_records.append({"model":key,"seed":seed,"artifact_path":str(p.relative_to(OUT_ROOT)).replace("\\","/"),
                              "artifact_state_sha256":state_hash,"files":files})

MODEL_ARTIFACT_MANIFEST={
    "schema_version":1,"stage_id":STAGE_ID,"protocol_version":PROTOCOL_VERSION,
    "frozen_package_sha256":EXPECTED_ZIP_SHA256,"decontamination_rule_version":DECONTAM_VERSION,
    "derived_clean_manifest_sha256":derived_manifest_sha,"cue_schema_sha256":CUE_SCHEMA["cue_schema_sha256"],
    "model_config_lock_sha256":MODEL_CONFIG_LOCK["config_lock_sha256"],"seeds":SEEDS,"models":model_records,
    "clean_split_artifacts":{s:{"path":str(p.relative_to(OUT_ROOT)).replace("\\","/"),"sha256":sha256_file(p)} for s,p in clean_paths.items()},
    "boundary":"No calibration, source TEST inference, or external inference occurred in Stage 03.",
}
MODEL_ARTIFACT_MANIFEST["manifest_payload_sha256"]=sha256_bytes(canonical_json_bytes(MODEL_ARTIFACT_MANIFEST))
write_json(OUT_ROOT/"MODEL_ARTIFACT_MANIFEST.json",MODEL_ARTIFACT_MANIFEST)

RUN_METADATA={
    "schema_version":1,"stage_id":STAGE_ID,"stage_name":STAGE_NAME,"protocol_version":PROTOCOL_VERSION,
    "execution_utc":utc_now(),"environment":environment_record(),"seeds":SEEDS,"config_split_seed":CONFIG_SPLIT_SEED,
    "input_identity":INPUT_IDENTITY,"firewall":{
        "D0_dev_test_read":"integrity/decontamination only","model_development":"CLEAN TRAIN only",
        "DEV":"not used after D0; reserved Stage 04 calibration","TEST":"not used after D0; reserved Stage 05",
        "external":"not read in Stage 03"},
}
write_json(OUT_ROOT/"RUN_METADATA.json",RUN_METADATA)

# Final reporting is derived from the persisted D0 split-summary artifact rather than
# transient DEV/TEST DataFrames. This keeps finalization independent of variables that
# were intentionally not materialized after the D0 firewall.
clean_summary_path = OUT_ROOT / "CLEAN_SPLIT_SUMMARY.csv"
if not clean_summary_path.is_file():
    hard_stop("Stage-03 clean split summary is missing during finalization", {"path": str(clean_summary_path)})
clean_summary_df = pd.read_csv(clean_summary_path)
required_summary_columns = {"split", "original_rows", "clean_rows", "quarantined_rows"}
missing_summary_columns = sorted(required_summary_columns - set(clean_summary_df.columns))
if missing_summary_columns:
    hard_stop("Stage-03 clean split summary has an unexpected schema", {"missing_columns": missing_summary_columns})
split_totals = clean_summary_df.loc[clean_summary_df["original_rows"].notna()].copy()
if set(split_totals["split"]) != {"train", "dev", "test"} or len(split_totals) != 3:
    hard_stop(
        "Stage-03 clean split summary does not contain exactly one aggregate row per protected split",
        {"observed_splits": split_totals["split"].astype(str).tolist(), "row_count": int(len(split_totals))},
    )
split_totals = split_totals.set_index("split")
clean_split_counts = {split: int(split_totals.loc[split, "clean_rows"]) for split in ["train", "dev", "test"]}
quarantined_row_count = int(split_totals["quarantined_rows"].sum())

summary_lines=[
    "# Stage 03 execution summary",
    "",
    f"- Protocol: `{PROTOCOL_VERSION}`",
    f"- Decontamination: `{DECONTAM_VERSION}`",
    f"- D0 status: **{D0_REPORT['status']}**",
    f"- Clean TRAIN/DEV/TEST rows: {clean_split_counts['train']} / {clean_split_counts['dev']} / {clean_split_counts['test']}",
    f"- Quarantined rows: {quarantined_row_count}",
    f"- Model panel: {', '.join(MODEL_KEYS)}",
    f"- Seeds: {SEEDS}",
    "- Firewall: model development used CLEAN TRAIN only after D0; DEV/TEST were not used for model decisions.",
]
write_text(OUT_ROOT/"STAGE03_SUMMARY.md","\n".join(summary_lines)+"\n")

# Hash every Stage-03 artifact except the checksum file itself and status (status written after hashing).
entries=[]
for p in sorted(OUT_ROOT.rglob("*"),key=str):
    if p.is_file() and p.name not in {"STAGE03_SHA256SUMS.txt","STAGE_STATUS.json"}:
        entries.append((str(p.relative_to(OUT_ROOT)).replace("\\","/"),sha256_file(p)))
write_text(OUT_ROOT/"STAGE03_SHA256SUMS.txt","\n".join(f"{h}  {rel}" for rel,h in entries)+"\n")

# Required artifact assertions.
required=["ENVIRONMENT_PREFLIGHT.json","INPUT_IDENTITY.json","FROZEN_INPUT_DISCOVERY.json","CLEAN_DATA_MANIFEST.json","D0_REPORT.json","DECONTAMINATION_FORENSICS.json","CLEAN_SPLIT_SUMMARY.csv",
          "CUE_FEATURE_SCHEMA.json","TRAIN_DEV_SPLIT_MANIFEST.json","MODEL_CONFIG_LOCK.json","MODEL_ARTIFACT_MANIFEST.json","RUN_METADATA.json","STAGE03_SUMMARY.md"]
missing=[x for x in required if not (OUT_ROOT/x).is_file()]
if missing: hard_stop("Stage-03 required artifact contract incomplete",{"missing":missing})
set_status("PASS","Stage-03 clean-view gate and source-only model development completed",
           {"output_root":str(OUT_ROOT),"model_count":len(model_records),"checksum_entries":len(entries)})
print("STAGE 03 COMPLETE — PASS")


STAGE 03 STATUS: PASS — Stage-03 clean-view gate and source-only model development completed
STAGE 03 COMPLETE — PASS
